In [1]:
!pip install seqeval

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.6/43.6 kB 3.3 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
  Created wheel for seqeval: filename=seqeval-1.2.2-py3-none-any.whl size=16162 sha256=036f0fe5e35c1bcc668e152aacf6fa90a7eb05b39b7b36626b70faa0fd36aff4
  Stored in directory: /root/.cache/pip/wheels/5f/b8/73/0b2c1a76b701a677653dd79ece07cfabd7457989dbfbdcd8d7
Successfully built seqeval


In [2]:
import pandas as pd
import ast
import torch
from datasets import Dataset
from sklearn.model_selection import train_test_split
from transformers import AutoTokenizer,AutoModelForTokenClassification, TrainingArguments, Trainer,T5Tokenizer, T5ForConditionalGeneration
from seqeval.metrics import classification_report, f1_score, precision_score, recall_score

In [3]:
df = pd.read_csv("https://docs.google.com/spreadsheets/d/1KXcA0PPOpygla1inEfnTc10FND6DFNL8p8vpQKAX6Tw/export?format=csv")

In [4]:
def normalize_triplets(x):
    if x is None:
        return "[]"
    x = str(x).strip()
    x = x.replace('""', '"')
    if x.startswith("'") and x.endswith("'"):
        x = x[1:-1]
    return x

def safe_parse(x):
    x = normalize_triplets(x)
    try:
        return ast.literal_eval(x)
    except:
        return []
df["triplets"] = df["triplets"].apply(safe_parse)
df = df[df["triplets"].apply(lambda x: len(x) > 0)]

In [5]:
train_df, test_df = train_test_split(
    df,
    test_size=0.1,
    random_state=42,
    shuffle=True
)
print(len(train_df), len(test_df))

2042 227


In [6]:
def create_labels(sentence, triplets):
    tokens = sentence.split()
    labels = ["O"] * len(tokens)

    for aspect, opinion, sentiment in triplets:
        asp_tokens = aspect.split()
        opi_tokens = opinion.split()

        if sentiment == 2:
            senti = "POS"
        elif sentiment == 0:
            senti = "NEG"
        else:
            senti = "NEU"

        # label aspect
        for i in range(len(tokens)):
            if tokens[i:i+len(asp_tokens)] == asp_tokens:
                labels[i] = "B-ASP"
                for j in range(1, len(asp_tokens)):
                    labels[i+j] = "I-ASP"

        # label opinion
        for i in range(len(tokens)):
            if tokens[i:i+len(opi_tokens)] == opi_tokens:
                labels[i] = f"B-OPI-{senti}"
                for j in range(1, len(opi_tokens)):
                    labels[i+j] = f"I-OPI-{senti}"

    return tokens, labels

In [7]:
label_list = [
    "O",
    "B-ASP", "I-ASP",
    "B-OPI-POS", "I-OPI-POS",
    "B-OPI-NEG", "I-OPI-NEG",
    "B-OPI-NEU", "I-OPI-NEU"
]

label2id = {l:i for i,l in enumerate(label_list)}
id2label = {i:l for l,i in label2id.items()}


def convert_df(df):
    all_tokens = []
    all_labels = []

    for _, row in df.iterrows():
        tokens, labels = create_labels(row["sentence_text"], row["triplets"])
        all_tokens.append(tokens)
        all_labels.append([label2id[l] for l in labels])

    return Dataset.from_dict({
        "tokens": all_tokens,
        "labels": all_labels
    })


train_dataset = convert_df(train_df)
test_dataset = convert_df(test_df)

In [8]:
tokenizer = AutoTokenizer.from_pretrained("bert-base-uncased")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

In [9]:
def tokenize_and_align(example):
    tokenized = tokenizer(
        example["tokens"],
        is_split_into_words=True,
        truncation=True,
        padding="max_length",
        max_length=128
    )

    word_ids = tokenized.word_ids()
    labels = example["labels"]

    new_labels = []
    prev = None

    for word_id in word_ids:
        if word_id is None:
            new_labels.append(-100)
        elif word_id != prev:
            new_labels.append(labels[word_id])
        else:
            new_labels.append(-100)
        prev = word_id

    tokenized["labels"] = new_labels
    return tokenized

In [10]:
train_dataset = train_dataset.map(tokenize_and_align)
test_dataset = test_dataset.map(tokenize_and_align)

Map:   0%|          | 0/2042 [00:00<?, ? examples/s]

Map:   0%|          | 0/227 [00:00<?, ? examples/s]

In [11]:
model = AutoModelForTokenClassification.from_pretrained(
    "bert-base-uncased",
    num_labels=len(label_list),
    id2label=id2label,
    label2id=label2id
)

model.safetensors:   0%|          | 0.00/440M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

BertForTokenClassification LOAD REPORT from: bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
bert.pooler.dense.bias                     | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
bert.pooler.dense.weight                   | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
classifier.weight                          | MISSING    | 
classifier.bias                            | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized be

In [12]:
def compute_metrics(pred):
    predictions, labels = pred
    predictions = predictions.argmax(axis=-1)

    true_labels = []
    pred_labels = []

    for p, l in zip(predictions, labels):
        true = []
        pred_ = []

        for pi, li in zip(p, l):
            if li != -100:
                true.append(id2label[li])
                pred_.append(id2label[pi])

        true_labels.append(true)
        pred_labels.append(pred_)

    return {
        "precision": precision_score(true_labels, pred_labels),
        "recall": recall_score(true_labels, pred_labels),
        "f1": f1_score(true_labels, pred_labels)
    }

In [22]:
training_args = TrainingArguments(
    output_dir="/content",
    learning_rate=2e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    num_train_epochs=5,
    eval_strategy = "epoch",
    save_strategy="epoch",
    logging_steps=50
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=test_dataset,
    compute_metrics=compute_metrics,
    processing_class = tokenizer
)

trainer.train()

Epoch,Training Loss,Validation Loss,Precision,Recall,F1
1,0.490740,0.599196,0.511137,0.636496,0.566970
2,0.371515,0.614743,0.561497,0.613139,0.586183
3,0.302248,0.645506,0.536465,0.633577,0.580991
4,0.262261,0.666470,0.567050,0.648175,0.604905
5,0.217590,0.684890,0.550251,0.639416,0.591492


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=640, training_loss=0.335993729531765, metrics={'train_runtime': 312.6699, 'train_samples_per_second': 32.654, 'train_steps_per_second': 2.047, 'total_flos': 667002181178880.0, 'train_loss': 0.335993729531765, 'epoch': 5.0})

In [14]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

In [ ]:
def extract_triplets(sentence):
    encoding = tokenizer(sentence, return_tensors="pt", truncation=True)

    word_ids = encoding.word_ids()
    inputs = {k: v.to(device) for k, v in encoding.items()}

    outputs = model(**inputs).logits
    preds = outputs.argmax(dim=-1)[0].tolist()

    tokens = tokenizer.convert_ids_to_tokens(inputs["input_ids"][0])

    aspects = []
    opinions = []
    current = []
    current_type = None
    sentiment = None

    for i, word_id in enumerate(word_ids):
        if word_id is None:
            continue
        token = tokens[i].replace("##", "")
        label = id2label[preds[i]]
        if label == "B-ASP":
            if current and current_type == "ASP":
                aspects.append(" ".join(current))
            current = [token]
            current_type = "ASP"
        elif label == "I-ASP" and current_type == "ASP":
            current.append(token)
        elif label.startswith("B-OPI"):
            if current and current_type == "OPI":
                opinions.append((" ".join(current), sentiment))
            current = [token]
            current_type = "OPI"
            if "POS" in label:
                sentiment = "POS"
            elif "NEG" in label:
                sentiment = "NEG"
            else:
                sentiment = "NEU"
        elif label.startswith("I-OPI") and current_type == "OPI":
            current.append(token)
        else:
            if current:
                if current_type == "ASP":
                    aspects.append(" ".join(current))
                else:
                    opinions.append((" ".join(current), sentiment))
            current = []
            current_type = None
    if current:
        if current_type == "ASP":
            aspects.append(" ".join(current))
        else:
            opinions.append((" ".join(current), sentiment))
    triplets = []
    for i in range(min(len(aspects), len(opinions))):
        triplets.append((aspects[i], opinions[i][0], opinions[i][1]))

    return triplets

In [23]:
sentence = "battery is great but screen is bad"

result = extract_triplets(sentence)

print(result)

[('battery', 'great', 'POS'), ('screen', 'bad', 'NEG')]
